In [11]:
import pandas as pd

df_tasks = pd.read_csv("CESAW_task_fact.csv")
df_wbs   = pd.read_csv("wbs_parent.csv")

# (Recommended) make sure df_wbs won't duplicate tasks during merge
df_wbs = df_wbs.drop_duplicates(subset=["project_key", "wbs_element_key"])

df_merged = df_tasks.merge(
    df_wbs[["project_key", "wbs_element_key", "parent_wbs_element_key"]],
    on=["project_key", "wbs_element_key"],
    how="left"
)

# Now: one row per task (same row count as df_tasks)
print(df_tasks.shape, df_merged.shape)

(61817, 12) (61817, 13)


In [12]:
df_tasks.head()

,project_key,person_key,team_key,wbs_element_key,plan_item_key,task_actual_start_date,task_actual_complete_date,task_actual_time_minutes,task_plan_time_minutes,phase_key,phase_short_name,process_name
0,100,296,31,11087,81765,10/29/2012 13:12,10/29/2012 13:20,7,90.0,337,DOC-Insp,Process-D
1,100,296,31,11088,81772,11/12/2012 4:43,11/12/2012 4:43,14,45.0,337,DOC-Insp,Process-D
2,100,296,31,11089,81747,11/20/2012 11:18,11/21/2012 2:16,101,30.0,337,DOC-Insp,Process-D
3,100,296,31,11089,81748,11/21/2012 2:16,11/21/2012 2:23,6,15.0,337,DOC-Insp,Process-D
4,100,296,31,11089,81749,11/20/2012 11:19,11/21/2012 2:23,76,15.0,337,DOC-Insp,Process-D


In [13]:
df_tasks = df_tasks.sort_values(by="plan_item_key", ascending=True)
df_tasks.head()

,project_key,person_key,team_key,wbs_element_key,plan_item_key,task_actual_start_date,task_actual_complete_date,task_actual_time_minutes,task_plan_time_minutes,phase_key,phase_short_name,process_name
8664,23,98,15,2583,20847,9/16/2011 1:31,9/16/2011 3:44,62,180.0,8,HLD,TSP(SM)
8665,23,98,15,2583,20848,9/16/2011 3:45,9/16/2011 3:48,3,60.0,10,HLD Review,TSP(SM)
8666,23,98,15,2583,20849,9/16/2011 3:48,9/16/2011 3:49,5,60.0,11,HLD Inspect,TSP(SM)
8626,23,96,15,2583,20849,9/16/2011 3:03,9/16/2011 3:11,8,60.0,11,HLD Inspect,TSP(SM)
8648,23,97,15,2583,20849,9/16/2011 3:30,9/16/2011 3:44,10,60.0,11,HLD Inspect,TSP(SM)


In [14]:
unique_tasks = df_tasks["plan_item_key"].nunique()
print("Unique number of tasks:", unique_tasks)

Unique number of tasks: 51582


In [15]:
df_tasks = (
    df_tasks
    .groupby("plan_item_key")
    .agg({
        "project_key": "first",
        "wbs_element_key": "first",
        "task_actual_start_date": "min",   # earliest work started
        "task_actual_complete_date": "max", # latest completion
        "task_actual_time_minutes": "sum",  # total effort
        "task_plan_time_minutes": "first",
        "person_key": "nunique" 
    })
    .reset_index()
)

In [16]:
df_tasks.head()

,plan_item_key,project_key,wbs_element_key,task_actual_start_date,task_actual_complete_date,task_actual_time_minutes,task_plan_time_minutes,person_key
0,20847,23,2583,9/16/2011 1:31,9/16/2011 3:44,62,180.000,1
1,20848,23,2583,9/16/2011 3:45,9/16/2011 3:48,3,60.000,1
2,20849,23,2583,9/16/2011 3:03,9/16/2011 3:49,23,60.000,3
3,20852,23,2587,9/16/2011 8:59,9/16/2011 9:02,3,11.412,1
4,20853,23,2587,9/16/2011 9:02,9/16/2011 9:09,7,22.305,1


In [17]:
target = "task_actual_time_minutes"

In [18]:
import numpy as np

# select numeric columns only
df_numeric = df_tasks.select_dtypes(include=[np.number])
corr_matrix = df_numeric.corr(method="pearson")
target_corr = (
    corr_matrix[target]
    .sort_values(ascending=False)
)

print(target_corr)

task_actual_time_minutes    1.000000
task_plan_time_minutes      0.553650
person_key                  0.249323
project_key                 0.029238
plan_item_key               0.026394
wbs_element_key            -0.000362
Name: task_actual_time_minutes, dtype: float64


In [ ]:
check = (
    df_merged
    .groupby("plan_item_key")["wbs_element_key"]
    .nunique()
    .reset_index(name="num_wbs_per_task")
)

In [24]:
multiple_wbs = check[check["num_wbs_per_task"] > 1]

print("Number of tasks linked to multiple WBS elements:", len(multiple_wbs))

Number of tasks linked to multiple WBS elements: 0


In [21]:
(df_merged.groupby("wbs_element_key")["plan_item_key"].nunique() > 1).any()

np.True_

In [22]:
df_merged.groupby("wbs_element_key")["plan_item_key"].nunique().describe()

count    7605.000000
mean        6.782643
std        96.610378
min         1.000000
25%         2.000000
50%         4.000000
75%         7.000000
max      7143.000000
Name: plan_item_key, dtype: float64

In [25]:
df_tasks = df_tasks.rename(columns={
    "person_key": "num_persons_per_task"
})

In [26]:
X = df_tasks[[
    "task_plan_time_minutes",
    "num_persons_per_task"
]]

y = df_tasks["task_actual_time_minutes"]

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [32]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)
model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=300, random_state=42)

In [33]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

MAE: 108.99449023873034
RMSE: 266.98902244057814


In [34]:
from sklearn.metrics import r2_score

print("R2:", r2_score(y_test, y_pred))

R2: 0.3803716787202438


In [46]:
df_merged["wbs_element_key"].nunique()

7605

In [47]:
df_merged = (
    df_merged
    .groupby("wbs_element_key")
    .agg({
        "project_key": "first",
        "task_actual_start_date": "min",
        "task_actual_complete_date": "max",
        "task_actual_time_minutes": "sum",
        "task_plan_time_minutes": "sum",  # better to sum at WBS level
        "person_key": "nunique",          # unique persons in this WBS
        "plan_item_key": "nunique"        # 👈 number of tasks in this WBS
    })
    .rename(columns={
        "person_key": "num_persons_in_wbs",
        "plan_item_key": "num_tasks_in_wbs"
    })
    .reset_index()
)

In [49]:
X = df_merged[[
    "task_plan_time_minutes",
    "num_persons_in_wbs",
    "num_tasks_in_wbs"
]]

y = df_merged["task_actual_time_minutes"]

In [50]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [51]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)
model.fit(X_train, y_train)


RandomForestRegressor(n_estimators=300, random_state=42)

In [52]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

MAE: 1594.1970188460741
RMSE: 32104.337937636497
R2: 0.05616731197164038
